In [ ]:
from pathlib import Path
import os

AGENT_KIND = "td3"  # "td3" | "sac"
RUN_MODE = "nominal"  # "nominal" | "disturb"
DISTURBANCE_PROFILE = "none"  # "none" | "ramp" | "fluctuation"
STATE_MODE = "standard"  # "standard" | "mismatch"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

DISTILLATION_DYN_PATH = None
DISTILLATION_SNAPS_PATH = None
DISTILLATION_VISIBLE = True


def resolve_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "utils").exists() and (candidate / "Data").exists():
            return candidate
    raise RuntimeError("Could not resolve the repo root from the current working directory.")


HERE = Path(os.getcwd())
REPO_ROOT = resolve_repo_root(HERE)


In [ ]:
import numpy as np
import torch

from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from TD3Agent.agent import TD3Agent
from utils.helpers import apply_min_max
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_weight_multiplier_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim
from utils.weights_runner import run_weight_multiplier_supervisor
from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_INPUT_BOUNDS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_RL_SETPOINTS_PHYS,
    DISTILLATION_SETPOINT_RANGE_PHYS,
    DISTILLATION_SS_INPUTS,
    RL_REWARD_DEFAULTS,
    WEIGHT_MULTIPLIER_BOUNDS,
    default_plant_paths,
)
from systems.distillation.data_io import (
    canonical_baseline_path,
    copy_legacy_distillation_data,
    load_distillation_system_data,
    resolve_distillation_data_dir,
    resolve_distillation_result_dir,
)
from systems.distillation.labels import DISTILLATION_SYSTEM_METADATA
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import (
    build_distillation_disturbance_schedule,
    canonical_disturbance_profile,
    validate_run_profile,
)


In [ ]:
validate_run_profile(RUN_MODE, DISTURBANCE_PROFILE)
DISTURBANCE_PROFILE = canonical_disturbance_profile(RUN_MODE, DISTURBANCE_PROFILE)

RUN_PROFILE_MAP = {
    ("td3", "nominal", "none"): {"n_tests": 200, "set_points_len": 200, "warm_start": 0, "test_cycle": [False, False, False, False, False]},
    ("td3", "disturb", "ramp"): {"n_tests": 200, "set_points_len": 200, "warm_start": 0, "test_cycle": [False, False, False, False, False]},
    ("td3", "disturb", "fluctuation"): {"n_tests": 200, "set_points_len": 200, "warm_start": 0, "test_cycle": [False, False, False, False, False]},
    ("sac", "nominal", "none"): {"n_tests": 200, "set_points_len": 200, "warm_start": 0, "test_cycle": [False, False, False, False, False]},
    ("sac", "disturb", "ramp"): {"n_tests": 200, "set_points_len": 200, "warm_start": 0, "test_cycle": [False, False, False, False, False]},
    ("sac", "disturb", "fluctuation"): {"n_tests": 200, "set_points_len": 200, "warm_start": 0, "test_cycle": [False, False, False, False, False]},
}
RUN_PROFILE = RUN_PROFILE_MAP[(AGENT_KIND, RUN_MODE, DISTURBANCE_PROFILE)]

copy_legacy_distillation_data(REPO_ROOT)
DATA_DIR = resolve_distillation_data_dir(REPO_ROOT)
RESULT_DIR = resolve_distillation_result_dir(REPO_ROOT)
DYN_PATH_DEFAULT, SNAPS_PATH_DEFAULT = default_plant_paths("weights", DISTURBANCE_PROFILE)
DYN_PATH = Path(DISTILLATION_DYN_PATH).expanduser() if DISTILLATION_DYN_PATH else DYN_PATH_DEFAULT
SNAPS_PATH = Path(DISTILLATION_SNAPS_PATH).expanduser() if DISTILLATION_SNAPS_PATH else SNAPS_PATH_DEFAULT

nominal_conditions = DISTILLATION_NOMINAL_CONDITIONS.copy()
ss_inputs = DISTILLATION_SS_INPUTS.copy()
u_min = DISTILLATION_INPUT_BOUNDS["u_min"].copy()
u_max = DISTILLATION_INPUT_BOUNDS["u_max"].copy()
setpoint_y = DISTILLATION_SETPOINT_RANGE_PHYS.copy()

y_sp_scenario_phys = DISTILLATION_RL_SETPOINTS_PHYS.copy()
system = build_distillation_system(
    path=DYN_PATH,
    ss_inputs=ss_inputs,
    initialization_point=nominal_conditions,
    delta_t=DELTA_T_HOURS,
    visible=DISTILLATION_VISIBLE,
)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:])

RESULT_PREFIX = f"distillation_weights_{AGENT_KIND}_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}_unified"
COMPARE_PREFIX = f"distillation_compare_weights_{AGENT_KIND}_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}"
BASELINE_MPC_PATH = canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE)
TOTAL_STEPS = RUN_PROFILE["n_tests"] * RUN_PROFILE["set_points_len"] * len(y_sp_scenario_phys)
DISTURBANCE_SCHEDULE = build_distillation_disturbance_schedule(RUN_MODE, DISTURBANCE_PROFILE, TOTAL_STEPS)


In [ ]:
poles = np.array([0.032, 0.03501095, 0.04099389, 0.04190188, 0.07477281, 0.01153274, 0.41036367])
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
STATE_DIM = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, STATE_MODE)
ACTION_DIM = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predict_h = 6
cont_h = 3
Q1_penalty = 1.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0
LOW_COEF = WEIGHT_MULTIPLIER_BOUNDS["low"].copy()
HIGH_COEF = WEIGHT_MULTIPLIER_BOUNDS["high"].copy()
ACTOR_FREEZE = 0

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)

reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    N_INPUTS,
    **RL_REWARD_DEFAULTS,
)
print(reward_params)

nominal_qi = 0.0
nominal_qs = 0.0
nominal_hA = 0.0
qi_change = 1.0
qs_change = 1.0
ha_change = 1.0

n_tests = RUN_PROFILE["n_tests"]
set_points_len = RUN_PROFILE["set_points_len"]
warm_start = RUN_PROFILE["warm_start"]
TEST_CYCLE = RUN_PROFILE["test_cycle"]

if AGENT_KIND == "td3":
    weight_agent = TD3Agent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=[512, 512, 512, 512, 512],
        critic_hidden=[512, 512, 512, 512, 512],
        gamma=0.995,
        actor_lr=1e-4,
        critic_lr=1e-4,
        batch_size=128,
        policy_delay=2,
        target_policy_smoothing_noise_std=0.1,
        noise_clip=0.2,
        max_action=1.0,
        tau=0.005,
        std_start=0.2,
        std_end=0.02,
        std_decay_rate=0.99995,
        std_decay_mode="exp",
        buffer_size=50_000,
        device=DEVICE,
        actor_freeze=ACTOR_FREEZE,
    )
elif AGENT_KIND == "sac":
    weight_agent = SACAgent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=[512, 512, 512, 512, 512],
        critic_hidden=[512, 512, 512, 512, 512],
        gamma=0.995,
        actor_lr=1e-4,
        critic_lr=1e-4,
        alpha_lr=1e-4,
        batch_size=128,
        grad_clip_norm=10.0,
        init_alpha=0.01,
        learn_alpha=True,
        target_entropy=-ACTION_DIM,
        target_update="soft",
        tau=0.005,
        hard_update_interval=10_000,
        activation="relu",
        use_layernorm=False,
        dropout=0.0,
        max_action=1.0,
        buffer_size=50_000,
        use_per=True,
        device=DEVICE,
        use_adamw=True,
        actor_freeze=ACTOR_FREEZE,
    )
else:
    raise ValueError("AGENT_KIND must be 'td3' or 'sac'.")

print(f"State dimension: {STATE_DIM}")
print(weight_agent.__class__.__name__)


In [ ]:
weight_cfg = {
    "agent_kind": AGENT_KIND,
    "run_mode": RUN_MODE,
    "state_mode": STATE_MODE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "low_coef": LOW_COEF,
    "high_coef": HIGH_COEF,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
}

runtime_ctx = {
    "system": system,
    "agent": weight_agent,
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "reward_params": reward_params,
    "system_stepper": distillation_system_stepper,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA.get("disturbance_labels"),
    "disturbance_schedule": DISTURBANCE_SCHEDULE,
}

result_bundle = run_weight_multiplier_supervisor(weight_cfg=weight_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH


In [ ]:
out_dir_rl = plot_weight_multiplier_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": DATA_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": 1,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_MODE,
    start_episode=1,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print(out_dir_rl)
print(out_dir_cmp)

try:
    system.close(SNAPS_PATH)
except Exception:
    pass
